In [1]:
import os
import sys
import subprocess
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.transforms import v2
import math
import torchvision.transforms.functional as TF
import torch.nn.functional as F



In [2]:
# Install torchinfo if not already available (Kaggle kernels usually have it pre‑installed)
try:
    from torchinfo import summary
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchinfo"])
    from torchinfo import summary



In [3]:
# Unzip the dataset using Python (avoids notebook‑only `!` syntax)
data_root = "/content/denoising_data"
os.makedirs(data_root, exist_ok=True)
shutil.unpack_archive("/kaggle/input/denoising-dirty-documents/train.zip", data_root)
shutil.unpack_archive("/kaggle/input/denoising-dirty-documents/test.zip", data_root)
shutil.unpack_archive(
    "/kaggle/input/denoising-dirty-documents/train_cleaned.zip", data_root
)



In [4]:
train_dir = os.path.join(data_root, "train")
train_cleaned_dir = os.path.join(data_root, "train_cleaned")
test_dir = os.path.join(data_root, "test")




In [5]:
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        if filename.lower().endswith(".png"):
            img_path = os.path.join(folder, filename)
            img = cv2.imread(img_path)  # BGR
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
    return images




In [6]:
train_images = load_images_from_folder(train_dir)
train_cleaned_images = load_images_from_folder(train_cleaned_dir)
test_images = load_images_from_folder(test_dir)



In [7]:
print(f"train_images: {len(train_images)}")
print(f"train_cleaned_images: {len(train_cleaned_images)}")
print(f"test_images: {len(test_images)}")



train_images: 115
train_cleaned_images: 115
test_images: 29


In [8]:
# (optional) inspect first image shapes
# print(train_images[0].shape, train_cleaned_images[0].shape, test_images[0].shape)



In [9]:
train_sizes = [img.shape[:2] for img in train_images]
train_cleaned_sizes = [img.shape[:2] for img in train_cleaned_images]
test_sizes = [img.shape[:2] for img in test_images]



In [10]:
unique_train_sizes = np.unique(train_sizes, axis=0)
unique_train_cleaned_sizes = np.unique(train_cleaned_sizes, axis=0)
unique_test_sizes = np.unique(test_sizes, axis=0)



In [11]:
print(f"train_images unique sizes: {unique_train_sizes}")
print(f"train_cleaned_images unique sizes: {unique_train_cleaned_sizes}")
print(f"test_images unique sizes: {unique_test_sizes}")




train_images unique sizes: [[258 540]
 [420 540]]
train_cleaned_images unique sizes: [[258 540]
 [420 540]]
test_images unique sizes: [[258 540]
 [420 540]]


In [12]:
class PadToSize:
    def __init__(self, target_size):
        self.target_size = target_size  # (height, width)

    def __call__(self, img):
        _, height, width = img.shape
        target_height, target_width = self.target_size
        pad_top = (target_height - height) // 2
        pad_bottom = target_height - height - pad_top
        pad_left = (target_width - width) // 2
        pad_right = target_width - width - pad_left
        return TF.pad(img, [pad_left, pad_top, pad_right, pad_bottom], fill=0)




In [13]:
class Grayscale:
    def __call__(self, img):
        pil_img = TF.to_pil_image(img) if isinstance(img, torch.Tensor) else img
        grayscale_img = pil_img.convert("L")
        return TF.to_tensor(grayscale_img)




In [14]:
train_transforms = v2.Compose(
    [
        v2.ToImage(),
        PadToSize((420, 540)),
        v2.RandomApply([v2.GaussianBlur(kernel_size=3)], p=0.4),
        v2.RandomApply([v2.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        Grayscale(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

val_transforms = v2.Compose(
    [
        v2.ToImage(),
        PadToSize((420, 540)),
        Grayscale(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

test_transforms = v2.Compose(
    [
        v2.ToImage(),
        PadToSize((420, 540)),
        Grayscale(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)




In [15]:
class ImageDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_files = sorted(
            [
                os.path.join(data_dir, f)
                for f in os.listdir(data_dir)
                if f.lower().endswith(".png")
            ]
        )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image




In [16]:
train_files = sorted(
    [
        os.path.join(train_dir, f)
        for f in os.listdir(train_dir)
        if f.lower().endswith(".png")
    ]
)
cleaned_files = sorted(
    [
        os.path.join(train_cleaned_dir, f)
        for f in os.listdir(train_cleaned_dir)
        if f.lower().endswith(".png")
    ]
)

train_files, val_files, cleaned_train, cleaned_val = train_test_split(
    train_files, cleaned_files, test_size=2 / 9, random_state=42
)




In [17]:
class PairedImageDataset(Dataset):
    def __init__(self, train_files, cleaned_files, transform=None):
        self.train_files = train_files
        self.cleaned_files = cleaned_files
        self.transform = transform

    def __len__(self):
        return len(self.train_files)

    def __getitem__(self, idx):
        train_img = Image.open(self.train_files[idx]).convert("RGB")
        cleaned_img = Image.open(self.cleaned_files[idx]).convert("RGB")
        if self.transform:
            train_img = self.transform(train_img)
            cleaned_img = self.transform(cleaned_img)
        return train_img, cleaned_img




In [18]:
train_dataset = PairedImageDataset(train_files, cleaned_train, train_transforms)
val_dataset = PairedImageDataset(val_files, cleaned_val, val_transforms)
test_dataset = ImageDataset(test_dir, test_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)




In [19]:
def visualize_paired_dataset(paired_loader, num_images=5):
    for train_imgs, cleaned_imgs in paired_loader:
        fig, axes = plt.subplots(num_images, 2, figsize=(8, num_images * 3))
        for i in range(num_images):
            axes[i, 0].imshow(train_imgs[i].permute(1, 2, 0).cpu().numpy(), cmap="gray")
            axes[i, 0].set_title(f"Original {i+1}")
            axes[i, 0].axis("off")
            axes[i, 1].imshow(
                cleaned_imgs[i].permute(1, 2, 0).cpu().numpy(), cmap="gray"
            )
            axes[i, 1].set_title(f"Cleaned {i+1}")
            axes[i, 1].axis("off")
        plt.tight_layout()
        plt.show()
        break


# Example visualisation (can be commented out to save time)
# visualize_paired_dataset(train_loader, num_images=3)
# visualize_paired_dataset(val_loader, num_images=2)



In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [21]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(1, 8, kernel_size=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(8, 8, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(8, 16, kernel_size=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.MaxPool2d(2),
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(16, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.MaxPool2d(2),
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(32, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.MaxPool2d(2),
        )
        self.enc5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.MaxPool2d(2),
        )
        self.dec5 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                64,
                64,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=64,
                bias=False,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                32,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=32,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                16,
                16,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=16,
                bias=False,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                8,
                8,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=8,
                bias=False,
            ),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(8, 1, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                1,
                1,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=1,
                bias=False,
            ),
            nn.Sigmoid(),
        )
        self.conv4 = nn.Conv2d(64, 32, kernel_size=1, bias=False)
        self.conv5 = nn.Conv2d(128, 64, kernel_size=1, bias=False)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        d5 = self.dec5(e5)
        d5 = F.interpolate(d5, size=(26, 33), mode="bilinear", align_corners=False)
        d5 = torch.cat([d5, e4], dim=1)
        d5 = self.conv5(d5)

        d4 = self.dec4(e4)
        d4 = F.interpolate(d4, size=(52, 67), mode="bilinear", align_corners=False)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = self.conv4(d4)

        d3 = self.dec3(d4)
        d3 = F.interpolate(d3, size=(105, 135), mode="bilinear", align_corners=False)

        d2 = self.dec2(d3)
        d2 = F.interpolate(d2, size=(210, 270), mode="bilinear", align_corners=False)

        d1 = self.dec1(d2)
        d1 = F.interpolate(d1, size=(420, 540), mode="bilinear", align_corners=False)

        return torch.clamp(d1, min=0.001, max=0.999)


model = DenoisingAutoencoder().to(device)



In [22]:
summary(model, input_size=(16, 1, 420, 540), device=device)




Layer (type:depth-idx)                   Output Shape              Param #
DenoisingAutoencoder                     [16, 1, 420, 540]         --
├─Sequential: 1-1                        [16, 8, 210, 270]         --
│    └─Conv2d: 2-1                       [16, 1, 420, 540]         9
│    └─Conv2d: 2-2                       [16, 8, 420, 540]         8
│    └─BatchNorm2d: 2-3                  [16, 8, 420, 540]         16
│    └─ReLU: 2-4                         [16, 8, 420, 540]         --
│    └─MaxPool2d: 2-5                    [16, 8, 210, 270]         --
├─Sequential: 1-2                        [16, 16, 105, 135]        --
│    └─Conv2d: 2-6                       [16, 8, 210, 270]         576
│    └─Conv2d: 2-7                       [16, 16, 210, 270]        128
│    └─BatchNorm2d: 2-8                  [16, 16, 210, 270]        32
│    └─ReLU: 2-9                         [16, 16, 210, 270]        --
│    └─Dropout: 2-10                     [16, 16, 210, 270]        --
│    └─MaxPool2

In [23]:
class RMSELoss(nn.Module):
    def forward(self, pred, target):
        return torch.sqrt(F.mse_loss(pred, target))




In [24]:
class HybridLoss(nn.Module):
    def __init__(self, lambda_rmse=0.8, lambda_l1=0.2):
        super().__init__()
        self.lambda_rmse = lambda_rmse
        self.lambda_l1 = lambda_l1
        self.rmse = RMSELoss()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        return self.lambda_rmse * self.rmse(pred, target) + self.lambda_l1 * self.l1(
            pred, target
        )


criterion = HybridLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)



In [25]:
epochs = 1000
patience = 5
early_stop_counter = 0
best_val_loss = float("inf")
best_model_state = None

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_rmse = 0.0
    for imgs, cleaned_imgs in train_loader:
        imgs, cleaned_imgs = imgs.to(device), cleaned_imgs.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, cleaned_imgs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_rmse += RMSELoss()(outputs, cleaned_imgs).item()
    train_loss /= len(train_loader)
    train_rmse /= len(train_loader)

    model.eval()
    val_loss = 0.0
    val_rmse = 0.0
    with torch.no_grad():
        for v_imgs, v_cleaned in val_loader:
            v_imgs, v_cleaned = v_imgs.to(device), v_cleaned.to(device)
            v_out = model(v_imgs)
            loss = criterion(v_out, v_cleaned)
            val_loss += loss.item()
            val_rmse += RMSELoss()(v_out, v_cleaned).item()
    val_loss /= len(val_loader)
    val_rmse /= len(val_loader)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} - "
        f"TrainLoss: {train_loss:.4f}, TrainRMSE: {train_rmse:.4f} | "
        f"ValLoss: {val_loss:.4f}, ValRMSE: {val_rmse:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        early_stop_counter = 0
        print("  New best model saved.")
    else:
        early_stop_counter += 1
        print(f"  No improvement. Early stop counter: {early_stop_counter}/{patience}")

    if early_stop_counter >= patience:
        print("Early stopping triggered.")
        break

if best_model_state is not None:
    torch.save(best_model_state, "best_model.pth")
    print("Best model checkpoint saved.")
else:
    print("No checkpoint to save.")



Epoch 1/1000 - TrainLoss: 0.4273, TrainRMSE: 0.4317 | ValLoss: 0.4738, ValRMSE: 0.4761
  New best model saved.
Epoch 2/1000 - TrainLoss: 0.4293, TrainRMSE: 0.4333 | ValLoss: 0.4705, ValRMSE: 0.4728
  New best model saved.
Epoch 3/1000 - TrainLoss: 0.3829, TrainRMSE: 0.3877 | ValLoss: 0.4630, ValRMSE: 0.4654
  New best model saved.
Epoch 4/1000 - TrainLoss: 0.3432, TrainRMSE: 0.3497 | ValLoss: 0.4511, ValRMSE: 0.4535
  New best model saved.
Epoch 5/1000 - TrainLoss: 0.3024, TrainRMSE: 0.3111 | ValLoss: 0.4355, ValRMSE: 0.4384
  New best model saved.
Epoch 6/1000 - TrainLoss: 0.2681, TrainRMSE: 0.2790 | ValLoss: 0.4253, ValRMSE: 0.4285
  New best model saved.
Epoch 7/1000 - TrainLoss: 0.2529, TrainRMSE: 0.2655 | ValLoss: 0.3683, ValRMSE: 0.3753
  New best model saved.
Epoch 8/1000 - TrainLoss: 0.2250, TrainRMSE: 0.2394 | ValLoss: 0.3280, ValRMSE: 0.3427
  New best model saved.
Epoch 9/1000 - TrainLoss: 0.2213, TrainRMSE: 0.2368 | ValLoss: 0.3142, ValRMSE: 0.3329
  New best model saved.
E

In [26]:
# Load best model for inference
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()



DenoisingAutoencoder(
  (enc1): Sequential(
    (0): Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(1, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (enc2): Sequential(
    (0): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.1, inplace=False)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (enc3): Sequential(
    (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(

In [27]:
# Generate predictions for the whole test set
all_outputs = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        preds = model(batch)  # (B,1,420,540)
        all_outputs.append(preds.cpu())
all_outputs = torch.cat(all_outputs, dim=0)  # (N_test,1,420,540)
print("All outputs shape:", all_outputs.shape)



All outputs shape: torch.Size([29, 1, 420, 540])


In [28]:
import csv

target_size = (420, 540)  # (height, width)


def compute_padding(orig_size, target_size):
    orig_h, orig_w = orig_size
    tgt_h, tgt_w = target_size
    pad_top = (tgt_h - orig_h) // 2 if tgt_h > orig_h else 0
    pad_left = (tgt_w - orig_w) // 2 if tgt_w > orig_w else 0
    return pad_top, pad_left


def remove_padding(pred_tensor, orig_size, target_size):
    orig_h, orig_w = orig_size
    pad_top, pad_left = compute_padding(orig_size, target_size)
    return pred_tensor[:, pad_top : pad_top + orig_h, pad_left : pad_left + orig_w]


# Build ordered list of test file paths (numeric order)
test_file_paths = sorted(
    [
        os.path.join(test_dir, f)
        for f in os.listdir(test_dir)
        if f.lower().endswith(".png")
    ],
    key=lambda x: int(os.path.splitext(os.path.basename(x))[0]),
)

submission_rows = []
for idx, path in enumerate(test_file_paths):
    img_id = os.path.splitext(os.path.basename(path))[0]

    # Original dimensions
    pil_img = Image.open(path).convert("L")
    orig_w, orig_h = pil_img.size
    orig_size = (orig_h, orig_w)

    # Corresponding model output (already on CPU)
    pred = all_outputs[idx]  # (1, 420, 540)
    cropped = remove_padding(pred, orig_size, target_size)  # (1, H, W)

    pred_np = cropped.squeeze(0).numpy()
    for r in range(orig_h):
        for c in range(orig_w):
            submission_rows.append([f"{img_id}_{r+1}_{c+1}", float(pred_np[r, c])])

# Write submission CSV to the Kaggle working directory
submission_path = "/kaggle/working/submission.csv"
with open(submission_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])
    writer.writerows(submission_rows)

print(f"Submission file written to {submission_path} with {len(submission_rows)} rows.")

Submission file written to /kaggle/working/submission.csv with 5789880 rows.
